For a single sample with logits z ∈ ℝ^C (C = number of classes) and true class y:
CrossEntropy(z, y) = -log( softmax(z)[y] )
softmax(z)[i] = exp(z[i]) / Σ_j exp(z[j])

CrossEntropy = -log( exp(z[y]) / Σ_j exp(z[j]) )
             = -z[y] + log( Σ_j exp(z[j]) )

This is called Log Sum Exp Form 

In [ ]:
# Naive Approach 
probs = softmax(logits)            # ← numerical disaster
loss = -log(probs[y])              # ← log(0) = -inf if probs[y] is tiny

#instead use log_softmax

## Without Label Smoothing

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def cross_entropy_loss(logits,targets):
    """
    logits: [N,C] 
    targets : [N]    
    """
    N,C = logits.shape 
    # log_softmax(z) = z - logsumexp(z)
    # This is equivalent to log(softmax(z)) but stable.
    log_probs = F.log_softmax(logits,dim=-1) # [N,C]
    # fancy indexing 
    gather = log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1) # [N]
    loss = -gather 
    return loss.mean()

## With Label Smoothing

Loss = -Σ_i target_smooth[i] * log_softmax(logits)[i].

For True class : Contribution = -(1 - ε) * log_softmax(z)[y].

For other classes : Contribution = -Σ_{i≠y} (ε / (C-1)) * log_softmax(z)[i].


Simplification by Pytorch
Loss_smooth = (1 - ε) * CE_hard + ε * CE_uniform

In [ ]:

def cross_entropy_with_smooth(logits,targets,smooth: int = 1e-3):
    N,C = logits.shape 
    log_probs = F.log_softmax(logits,dim=-1) # N,C
  # CE_Hard label loss 
    gather = log_probs.gather(dim=-1,index=targets.unsqueeze(1)).squeeze(1)# [N]
    log_mean = -log_probs.mean(dim=-1) # N
    loss = -(1.0-smooth)*gather + smooth*log_mean
    return loss.mean()